# Syllable use per mouse (supplementary)

fig_segmentation panel K — the syllable composition over the time-warped trial, one stacked column per bin (paw states pale/still → dark/vigorous; whisking and licking hatched) — drawn for **every one of the 56 mice** in the LDA cohort, from all trials of the mouse's sessions in that cohort. Mice are in a **random order** (fixed seed), so the figure shows the range of individual patterns without implying an axis.

Same syllable file and the same drawing function (`paper_style.syllable_histogram`) as fig_segmentation, so the panels are directly comparable with K.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

FIGDIR = pathlib.Path.cwd().resolve()
while not (FIGDIR / 'fig_layout.py').exists() and FIGDIR != FIGDIR.parent:
    FIGDIR = FIGDIR / 'figure_generation' if (FIGDIR / 'figure_generation').exists() else FIGDIR.parent
PI = FIGDIR.parent
for p in [str(PI), str(FIGDIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)
try:
    import numpy._core.numeric  # noqa: F401
except ModuleNotFoundError:     # the LDA file was pickled under numpy 2
    import numpy.core.numeric, numpy.core.multiarray
    sys.modules['numpy._core.numeric'], sys.modules['numpy._core.multiarray'] = np.core.numeric, np.core.multiarray
import paper_style as ps
import fig_layout as fl
ps.use('paper')
ps.use_paw_states('uniform')
FIG = 'fig_syllable_use_per_mouse'

SYLLABLE_FILE = next(f for f in [PI / 'data' / '8_k_10_bin_syllables_02-10-2026',                  # fig_segmentation's
                                 PI / 'clustering' / 'data_files' / '8_k_10_bin_syllables_02-10-2026'] if f.exists())
LDA_FILE = PI / 'clustering' / 'data_files' / 'mouse_LDA_5_bins_raw_shrink0.5_wmouse_360_02-10-2026'
TEXTURE = True          # whisk / lick hatching, as panel K; False = paw colour only (cleaner at this size)
N_COLS = 8
SEED = 0           # the (random) order of the mice
EPOCHS = ps.EPOCH_NAMES

## Trials per mouse, mice in a random order

In [ ]:
lda = pd.read_pickle(LDA_FILE).rename(columns={0: 'LD1'})
seq = pd.read_parquet(SYLLABLE_FILE)
seq['session'] = seq['sample'].str[:36]
seq = seq[seq['session'].isin(lda['session'])].copy()
seq['binned_sequence'] = seq['binned_sequence'].apply(ps.paw_codes)     # -> uniform (vigor) numbering, as fig_segmentation

def codes_for(rows):
    piv = rows.pivot(index=['sample'], columns=['broader_label'], values='binned_sequence').dropna()
    return np.vstack(piv[EPOCHS].apply(lambda r: np.hstack(r), axis=1)).astype(float)

MICE = list(np.random.default_rng(SEED).permutation(sorted(lda['mouse_name'].unique())))
CODES = {m: codes_for(seq[seq['mouse_name'] == m]) for m in MICE}
print(f"{SYLLABLE_FILE.name}: {len(MICE)} mice, {seq['session'].nunique()} sessions, "
      f"{sum(len(c) for c in CODES.values()):,} trials (per mouse: {min(len(c) for c in CODES.values())}-"
      f"{max(len(c) for c in CODES.values())})")

## The figure

In [ ]:
n_rows = int(np.ceil(len(MICE) / N_COLS))
fig = plt.figure(figsize=(fl.FIG_WIDTH, 1.02 * n_rows + 0.55))
G = fig.add_gridspec(n_rows, N_COLS, wspace=0.12, hspace=0.42, left=0.07, right=0.99, top=0.975,
                     bottom=0.55 / (1.02 * n_rows + 0.55) + 0.03)
axes = []
for i, m in enumerate(MICE):
    r_, c_ = divmod(i, N_COLS)
    ax = fig.add_subplot(G[r_, c_]); axes.append(ax)
    ps.syllable_histogram(CODES[m], ax=ax, texture=TEXTURE, warn=False)
    ps.epoch_lines(ax, CODES[m].shape[1])
    ax.set_title(f'{m}  (n = {len(CODES[m])})', fontsize=plt.rcParams['font.size'] * 0.62, pad=2)
    ax.set_ylabel('')
    ax.set_box_aspect(1)
    ax.tick_params(length=1.5, labelsize=plt.rcParams['xtick.labelsize'] * 0.7)
    if c_ == 0:
        ax.set_yticks([0, 1]); ax.set_ylabel('Fraction', fontsize=plt.rcParams['font.size'] * 0.7, labelpad=0)
    else:
        ax.set_yticks([])
    last_row = i + N_COLS >= len(MICE)
    if last_row:
        ax.set_xticks([5, 15, 25, 35]); ax.set_xticklabels(ps.EPOCH_SHORT, rotation=45, ha='right')
    else:
        ax.set_xticks([])
# key: paw states (with their side) + the whisk / lick hatching
ps.paw_legend(fig, ncol=6, hatch=TEXTURE, loc='lower left', bbox_to_anchor=(0.06, 0.0),
              fontsize=plt.rcParams['legend.fontsize'] * 0.8)
plt.show()
OUT = FIGDIR / 'figures'
for path in [OUT / f"{FIG}_{date.today().strftime('%d-%m-%Y')}.svg", OUT / f'{FIG}.pdf', OUT / f'{FIG}.png']:
    fig.savefig(path, dpi=300)
    print('wrote', path.relative_to(PI))